# Email Dataset Preprocessing Notebook

This notebook preprocesses multiple email datasets for an AI email intelligence project. It loads raw CSV files, standardizes schema, cleans text, visualizes class distributions, and exports cleaned datasets.

In [ ]:
# Core imports
from pathlib import Path
import re

import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
sns.set_theme(style='whitegrid')

print('pandas version :', pd.__version__)
print('numpy version  :', np.__version__)
print('sklearn version:', sklearn.__version__)

In [ ]:
# Define project paths and ensure output folders exist
BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / 'data' / 'raw'
CLEAN_DIR = BASE_DIR / 'data' / 'clean'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

DATASET_FILES = {
    'aa_dataset': RAW_DIR / 'aa_dataset.csv',
    'CEAS_08': RAW_DIR / 'CEAS_08.csv',
    'mail_data': RAW_DIR / 'mail_data.csv',
}

missing_files = [str(path) for path in DATASET_FILES.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        'The following required dataset files were not found:\n' + '\n'.join(missing_files)
    )

RAW_DIR, CLEAN_DIR

In [ ]:
# Helper functions for loading, cleaning, and schema normalization
def load_dataset(path: Path) -> pd.DataFrame:
    return pd.read_csv(path)


def display_dataset_preview(name: str, df: pd.DataFrame) -> None:
    print(f'\n{'=' * 80}')
    print(f'{name} column names:')
    print(df.columns.tolist())
    print(f'\n{name} first 5 rows:')
    display(df.head())


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    clean_cols = []
    for col in df.columns:
        normalized = re.sub(r'[^a-z0-9]+', '_', str(col).strip().lower()).strip('_')
        clean_cols.append(normalized)
    out = df.copy()
    out.columns = clean_cols
    return out


def remove_html_tags(text: str) -> str:
    return re.sub(r'<[^>]+>', ' ', text)


def clean_text(text) -> str:
    if pd.isna(text):
        return np.nan
    text = str(text).lower()
    text = remove_html_tags(text)
    text = re.sub(r'&nbsp;|&amp;|&lt;|&gt;', ' ', text)
    text = re.sub(r'[^a-z0-9\s\.,!\?@:_\-/]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def clean_text_columns(df: pd.DataFrame, text_columns) -> pd.DataFrame:
    out = df.copy()
    for col in text_columns:
        if col in out.columns:
            out[col] = out[col].apply(clean_text)
    return out


def drop_missing_text_rows(df: pd.DataFrame, required_columns) -> pd.DataFrame:
    valid_columns = [col for col in required_columns if col in df.columns]
    out = df.copy()
    if valid_columns:
        out[valid_columns] = out[valid_columns].replace(r'^\s*$', np.nan, regex=True)
        out = out.dropna(subset=valid_columns)
    return out


def find_first_existing(columns, candidates):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None


def plot_distribution(df: pd.DataFrame, column: str, title: str) -> None:
    counts = df[column].astype(str).value_counts(dropna=False)
    plt.figure(figsize=(8, 4))
    ax = sns.barplot(x=counts.index, y=counts.values, palette='Blues_d')
    ax.set_title(title)
    ax.set_xlabel(column)
    ax.set_ylabel('Count')
    for idx, value in enumerate(counts.values):
        ax.text(idx, value + max(counts.values) * 0.01, str(value), ha='center', va='bottom')
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

In [ ]:
# Load all raw datasets
raw_datasets = {name: load_dataset(path) for name, path in DATASET_FILES.items()}

for dataset_name, dataset_df in raw_datasets.items():
    display_dataset_preview(dataset_name, dataset_df)

## Preprocess `aa_dataset`

Keep `subject`, `body`, `answer`, `priority`, and `type`, then standardize names and clean text.

In [ ]:
aa_raw = normalize_columns(raw_datasets['aa_dataset'])

aa_subject_col = find_first_existing(aa_raw.columns, ['subject', 'subj', 'email_subject'])
aa_body_col = find_first_existing(aa_raw.columns, ['body', 'message', 'email_body', 'content', 'text'])
aa_answer_col = find_first_existing(aa_raw.columns, ['answer', 'response', 'reply'])
aa_priority_col = find_first_existing(aa_raw.columns, ['priority', 'urgency'])
aa_type_col = find_first_existing(aa_raw.columns, ['type', 'category', 'ticket_type'])

required_aa_cols = [aa_subject_col, aa_body_col, aa_answer_col, aa_priority_col, aa_type_col]
if any(col is None for col in required_aa_cols):
    raise ValueError(
        'aa_dataset.csv is missing one or more required columns. '
        f'Available columns: {aa_raw.columns.tolist()}'
    )

aa_df = aa_raw[[aa_subject_col, aa_body_col, aa_answer_col, aa_priority_col, aa_type_col]].copy()
aa_df = aa_df.rename(
    columns={
        aa_subject_col: 'subject',
        aa_body_col: 'body',
        aa_answer_col: 'answer',
        aa_priority_col: 'priority',
        aa_type_col: 'type',
    }
)

aa_before = len(aa_df)
aa_df = aa_df.drop_duplicates()
aa_df = drop_missing_text_rows(aa_df, ['subject', 'body', 'answer'])
aa_df = clean_text_columns(aa_df, ['subject', 'body', 'answer'])
aa_df = drop_missing_text_rows(aa_df, ['subject', 'body', 'answer'])
aa_df['text'] = (aa_df['subject'].fillna('') + ' ' + aa_df['body'].fillna('')).str.strip()
aa_df = drop_missing_text_rows(aa_df, ['text'])

print(f'aa_dataset rows before cleaning: {aa_before}')
print(f'aa_dataset rows after cleaning : {len(aa_df)}')
display(aa_df.head())

## Preprocess `CEAS_08`

Combine `subject` and `body` into `text`, then create a unified `spam_label` column.

In [ ]:
ceas_raw = normalize_columns(raw_datasets['CEAS_08'])

ceas_subject_col = find_first_existing(ceas_raw.columns, ['subject', 'subj'])
ceas_body_col = find_first_existing(ceas_raw.columns, ['body', 'message', 'content', 'text'])
ceas_label_col = find_first_existing(ceas_raw.columns, ['label', 'class', 'spam', 'spam_label'])

required_ceas_cols = [ceas_subject_col, ceas_body_col, ceas_label_col]
if any(col is None for col in required_ceas_cols):
    raise ValueError(
        'CEAS_08.csv is missing one or more required columns. '
        f'Available columns: {ceas_raw.columns.tolist()}'
    )

ceas_df = ceas_raw[[ceas_subject_col, ceas_body_col, ceas_label_col]].copy()
ceas_df = ceas_df.rename(
    columns={
        ceas_subject_col: 'subject',
        ceas_body_col: 'body',
        ceas_label_col: 'spam_label',
    }
)

ceas_before = len(ceas_df)
ceas_df = ceas_df.drop_duplicates()
ceas_df = drop_missing_text_rows(ceas_df, ['subject', 'body', 'spam_label'])
ceas_df = clean_text_columns(ceas_df, ['subject', 'body'])
ceas_df['text'] = (ceas_df['subject'].fillna('') + ' ' + ceas_df['body'].fillna('')).str.strip()
ceas_df = drop_missing_text_rows(ceas_df, ['text'])
ceas_df['spam_label'] = (
    ceas_df['spam_label']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({'spam': 1, 'ham': 0, 'nonspam': 0, 'not spam': 0})
)
ceas_df['spam_label'] = pd.to_numeric(ceas_df['spam_label'], errors='coerce')
ceas_df = ceas_df.dropna(subset=['spam_label'])
ceas_df['spam_label'] = ceas_df['spam_label'].astype(int)
ceas_df = ceas_df[['subject', 'body', 'text', 'spam_label']].reset_index(drop=True)

print(f'CEAS_08 rows before cleaning: {ceas_before}')
print(f'CEAS_08 rows after cleaning : {len(ceas_df)}')
display(ceas_df.head())

## Preprocess `mail_data`

Rename label-style columns into `spam_label` and clean the message text.

In [ ]:
mail_raw = normalize_columns(raw_datasets['mail_data'])

mail_text_col = find_first_existing(mail_raw.columns, ['text', 'message', 'body', 'content'])
mail_label_col = find_first_existing(mail_raw.columns, ['spam_label', 'label', 'category', 'class', 'target'])

required_mail_cols = [mail_text_col, mail_label_col]
if any(col is None for col in required_mail_cols):
    raise ValueError(
        'mail_data.csv is missing one or more required columns. '
        f'Available columns: {mail_raw.columns.tolist()}'
    )

mail_df = mail_raw[[mail_text_col, mail_label_col]].copy()
mail_df = mail_df.rename(columns={mail_text_col: 'text', mail_label_col: 'spam_label'})

mail_before = len(mail_df)
mail_df = mail_df.drop_duplicates()
mail_df = drop_missing_text_rows(mail_df, ['text', 'spam_label'])
mail_df = clean_text_columns(mail_df, ['text'])
mail_df['spam_label'] = (
    mail_df['spam_label']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({'spam': 1, 'ham': 0, 'nonspam': 0, 'not spam': 0})
)
mail_df['spam_label'] = pd.to_numeric(mail_df['spam_label'], errors='coerce')
mail_df = mail_df.dropna(subset=['text', 'spam_label'])
mail_df['spam_label'] = mail_df['spam_label'].astype(int)
mail_df = mail_df.reset_index(drop=True)

print(f'mail_data rows before cleaning: {mail_before}')
print(f'mail_data rows after cleaning : {len(mail_df)}')
display(mail_df.head())

## Create final clean datasets

Build task-specific outputs for triage, spam classification, and reply generation.

In [ ]:
# Triage dataset from aa_dataset
triage_dataset = aa_df[['subject', 'body', 'text', 'priority', 'type']].copy()

# Reply dataset from aa_dataset
reply_dataset = aa_df[['subject', 'body', 'text', 'answer', 'priority', 'type']].copy()

# Spam dataset from CEAS_08 and mail_data
spam_dataset = pd.concat(
    [
        ceas_df[['text', 'spam_label']].copy(),
        mail_df[['text', 'spam_label']].copy(),
    ],
    ignore_index=True,
)
spam_dataset = spam_dataset.drop_duplicates().reset_index(drop=True)

print('triage_dataset shape:', triage_dataset.shape)
print('reply_dataset shape :', reply_dataset.shape)
print('spam_dataset shape  :', spam_dataset.shape)

display(triage_dataset.head())
display(reply_dataset.head())
display(spam_dataset.head())

## Class distribution charts

In [ ]:
plot_distribution(triage_dataset, 'priority', 'Triage Dataset Priority Distribution')
plot_distribution(triage_dataset, 'type', 'Triage Dataset Type Distribution')
plot_distribution(spam_dataset, 'spam_label', 'Spam Dataset Class Distribution')

## Save cleaned outputs

In [ ]:
triage_output = CLEAN_DIR / 'triage_dataset.csv'
spam_output = CLEAN_DIR / 'spam_dataset.csv'
reply_output = CLEAN_DIR / 'reply_dataset.csv'

triage_dataset.to_csv(triage_output, index=False)
spam_dataset.to_csv(spam_output, index=False)
reply_dataset.to_csv(reply_output, index=False)

print('Saved files:')
print(triage_output)
print(spam_output)
print(reply_output)